# Gradient Checkpointing

> Parte da série [ML Notebooks](../README.md) — por **Nandobez**.


## Intuição

Ativações dominam a memória em redes profundas. Gradient checkpointing descarta ativações no forward e recomputa no backward, trocando cerca de 33 % mais compute por uma queda grande de memória. Permite treinar modelos que não caberiam.


## Formulação Matemática

Memória: $O(N) \to O(\sqrt{N})$ ao checkpointar a cada $\sqrt{N}$ camadas (checkpointing segmentado).


## Implementação


In [ ]:
import torch
import torch.nn as nn
from torch.utils.checkpoint import checkpoint


In [ ]:
class Block(nn.Module):
    def __init__(self, d):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(d, d), nn.GELU(), nn.Linear(d, d))
    def forward(self, x): return self.net(x) + x

class Stack(nn.Module):
    def __init__(self, d=512, n=8, use_ckpt=False):
        super().__init__()
        self.blocks = nn.ModuleList([Block(d) for _ in range(n)])
        self.use_ckpt = use_ckpt
    def forward(self, x):
        for b in self.blocks:
            x = checkpoint(b, x, use_reentrant=False) if self.use_ckpt else b(x)
        return x


## Experimento


In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
x = torch.randn(32, 512, device=device, requires_grad=True)

def measure(model):
    if device == 'cuda':
        torch.cuda.reset_peak_memory_stats()
    y = model(x).sum()
    y.backward()
    peak = torch.cuda.max_memory_allocated() / 1e6 if device == 'cuda' else None
    return peak

for use_ckpt in [False, True]:
    model = Stack(d=512, n=8, use_ckpt=use_ckpt).to(device)
    peak = measure(model)
    print(f'use_ckpt={use_ckpt}  peak={peak} MB')


## Discussão

- Checkpointar um único módulo é o começo mais fácil; para controle fino, segmente a rede em grupos.
- Checkpointing reentrante é a API antiga; use `use_reentrant=False` para o dispatcher moderno.
- A economia compõe com precisão mista e offloading (e.g. DeepSpeed ZeRO).


## Referências

- Repositório da série: [github.com/Nandobez/ml-notebooks](https://github.com/Nandobez/ml-notebooks)
- Autor: [Nandobez](https://github.com/Nandobez)
